# Part 2: Data setup

## Task 1 Cell 1

In [ ]:
%pip install -r ../src/requirements.txt

## Task 1 Cell 2

In [1]:
# Standard library
import os
from pathlib import Path
import time

# Database driver
import psycopg

# Environment variable loading
from dotenv import load_dotenv


## Task 1 Cell 3

In [ ]:
env_path = next(
    (
        directory / "src" / ".env"
        for directory in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (directory / "src" / ".env").exists()
    ),
    None,
)
if env_path is None:
    raise FileNotFoundError("Could not find src/.env. Copy src/.env.sample to src/.env and add your lab values.")
load_dotenv(dotenv_path=env_path, override=True)

# Part 2.5 registers these deployments with azure_ai. Keep the key in memory
# only; do not print it or interpolate it into SQL text.
AZURE_OPENAI_ENDPOINT   = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_DEPLOYMENT = os.environ["AZURE_OPENAI_DEPLOYMENT"]
AZURE_OPENAI_KEY        = os.environ["AZURE_OPENAI_KEY"]
AZURE_EMBED_DEPLOYMENT  = os.environ["AZURE_EMBED_DEPLOYMENT"]
AZURE_API_VERSION       = None

DB_CONFIG = {
    "host":     os.environ["AZURE_PG_HOST"],
    "dbname":   os.environ["AZURE_PG_NAME"],
    "user":     os.environ["AZURE_PG_USER"],
    "password": os.environ["AZURE_PG_PASSWORD"],
    "port":     os.environ["AZURE_PG_PORT"],
    "sslmode":  os.environ.get("AZURE_PG_SSLMODE", "require"),
}

# ── Stable lab object names, used by every later cell in this notebook ──
# Keep these in sync with the matching block in Notebook 2.
SOURCE_TABLE  = "public.cases"                # metadata, opinions, BM25, graph, azure_ai.extract
CHUNK_TABLE   = "public.case_opinion_chunks"  # AI Pipeline sink: one row per opinion chunk
PIPELINE_NAME = "case_opinion_embedding_pipeline"
EMBED_MODEL   = "lab-embedding"               # registered in Part 2.5
CHAT_MODEL    = "lab-chat"                    # registered in Part 2.5
EMBED_DIMS    = 1536                          # text-embedding-3-small output size

print(f"Host:            {DB_CONFIG['host']}")
print(f"Database:        {DB_CONFIG['dbname']}")
print(f"User:            {DB_CONFIG['user']}")
print(f"SSL mode:        {DB_CONFIG['sslmode']}")
print(f"OpenAI endpoint: {AZURE_OPENAI_ENDPOINT}")
print(f"Chat deployment: {AZURE_OPENAI_DEPLOYMENT}")
print(f"Embed deployment:{AZURE_EMBED_DEPLOYMENT}")
print(f"API version:     {AZURE_API_VERSION}")
print(f"Source table:    {SOURCE_TABLE}")
print(f"Chunk sink:      {CHUNK_TABLE}")
print(f"Pipeline:        {PIPELINE_NAME}")
print(f"Embedding model: {EMBED_MODEL} ({EMBED_DIMS} dims)")
print(f"Chat model:      {CHAT_MODEL}")


## Task 2 Cell 1

In [ ]:
# Verify and install the extensions needed for the lab.
REQUIRED_EXTENSIONS = [
    ("azure_ai",      "in-database AI functions + the ai.* AI Pipeline API"),
    ("vector",        "the vector data type and similarity operators"),
    ("age",           "Apache AGE property graph (case_graph)"),
    ("pg_diskann",    "DiskANN ANN index with advanced filtering"),
    ("pg_textsearch", "BM25 full-text index"),
]

OPTIONAL_EXTENSIONS = [
    ("pg_durable", "durable AI Pipeline run history in df.instances"),
]

REQUIRED_PRELOADS = {"age", "pg_textsearch"}
OPTIONAL_PRELOADS = {"pg_durable"}


def parse_setting(value):
    return {item.strip() for item in (value or "").split(",") if item.strip()}


with psycopg.connect(**DB_CONFIG, autocommit=True) as conn, conn.cursor() as cur:
    cur.execute("SELECT extname FROM pg_extension;")
    installed = {row[0] for row in cur.fetchall()}

    cur.execute("SELECT name FROM pg_available_extensions;")
    available = {row[0] for row in cur.fetchall()}

    cur.execute(
        "SELECT current_setting('azure.extensions', true), "
        "current_setting('shared_preload_libraries', true);"
    )
    allowed_value, preloaded_value = cur.fetchone()
    allowed = parse_setting(allowed_value)
    preloaded = parse_setting(preloaded_value)

    print(f"azure.extensions: {allowed_value or '(empty)'}")
    print(f"shared_preload_libraries: {preloaded_value or '(empty)'}")

    unavailable = [
        ext for ext, _ in REQUIRED_EXTENSIONS
        if ext not in installed and ext not in available
    ]
    if unavailable:
        raise RuntimeError(
            "This HorizonDB cluster doesn't offer required extensions: "
            + ", ".join(unavailable)
        )

    missing_preloads = sorted(REQUIRED_PRELOADS - preloaded)
    if missing_preloads:
        raise RuntimeError(
            "Update the connected HorizonDB parameter group: add "
            + ", ".join(missing_preloads)
            + " to shared_preload_libraries, reconnect the parameter group, "
              "wait for the cluster restart, and rerun this cell."
        )

    missing_allowlist = [
        ext for ext, _ in REQUIRED_EXTENSIONS
        if ext not in installed and ext not in allowed
    ]
    if missing_allowlist:
        raise RuntimeError(
            "Update the connected HorizonDB parameter group: add "
            + ", ".join(missing_allowlist)
            + " to azure.extensions, reconnect the parameter group, and rerun this cell."
        )

    for ext, why in REQUIRED_EXTENSIONS:
        if ext in installed:
            print(f"✅ {ext:<14} ready   - already installed; {why}")
            continue

        cur.execute(f"CREATE EXTENSION {ext} CASCADE;")
        installed.add(ext)
        print(f"✅ {ext:<14} ready   - installed now; {why}")

    for ext, why in OPTIONAL_EXTENSIONS:
        if ext in installed:
            print(f"✅ {ext:<14} ready   - already installed; {why}")
        elif ext not in allowed:
            print(
                f"⚠️  {ext:<14} skipped - add it to azure.extensions "
                "to enable optional durable-run history."
            )
        elif ext in OPTIONAL_PRELOADS and ext not in preloaded:
            print(
                f"⚠️  {ext:<14} skipped - add it to shared_preload_libraries "
                "and restart the cluster first."
            )
        else:
            cur.execute(f"CREATE EXTENSION {ext} CASCADE;")
            installed.add(ext)
            print(f"✅ {ext:<14} ready   - installed now; {why}")

    cur.execute(
        '''
        SELECT extname, extversion
        FROM pg_extension
        WHERE extname IN (
            'azure_ai', 'vector', 'age', 'pg_diskann', 'pg_textsearch', 'pg_durable'
        )
        ORDER BY extname;
        '''
    )
    print("\nInstalled extension versions:")
    for name, version in cur.fetchall():
        print(f"  {name:<14} {version}")

print("\nDone. Required extensions are ready.")


## Task 2 Cell 2

In [ ]:
# Register Azure OpenAI deployments in the azure_ai model registry
CHAT_MODEL_ALIAS  = "lab-chat"
EMBED_MODEL_ALIAS = "lab-embedding"

conn = psycopg.connect(**DB_CONFIG)
conn.autocommit = True

with conn.cursor() as cur:
    # Remove any prior registrations so this cell is safely re-runnable
    for alias in (CHAT_MODEL_ALIAS, EMBED_MODEL_ALIAS):
        try:
            cur.execute("SELECT model_registry.model_remove(%s);", (alias,))
        except Exception:
            pass

    # Register the chat-completion deployment
    cur.execute(
        """
        SELECT model_registry.model_add(
            %s,                 -- model alias
            %s,                 -- endpoint URL
            %s,                 -- deployment name
            %s,                 -- model name
            %s,                 -- API version
            'subscription-key', -- auth type
            %s                  -- endpoint key
        );
        """,
        (
            CHAT_MODEL_ALIAS,
            AZURE_OPENAI_ENDPOINT,
            AZURE_OPENAI_DEPLOYMENT,
            AZURE_OPENAI_DEPLOYMENT,
            AZURE_API_VERSION,
            AZURE_OPENAI_KEY,
        ),
    )
    print(f"Registered chat model alias:      {CHAT_MODEL_ALIAS}")

    # Register the embedding deployment
    cur.execute(
        """
        SELECT model_registry.model_add(
            %s, %s, %s, %s, %s, 'subscription-key', %s
        );
        """,
        (
            EMBED_MODEL_ALIAS,
            AZURE_OPENAI_ENDPOINT,
            AZURE_EMBED_DEPLOYMENT,
            AZURE_EMBED_DEPLOYMENT,
            AZURE_API_VERSION,
            AZURE_OPENAI_KEY,
        ),
    )
    print(f"Registered embedding model alias: {EMBED_MODEL_ALIAS}")

    # Verify the registrations
    cur.execute(
        """
        SELECT alias, endpoint, deployment_name, model_name, api_version, auth_type        
        FROM model_registry.model_list_all()        
        ORDER BY alias;
        """        
    )
    for row in cur.fetchall():
        print(row)

conn.close()


## Task 2 Cell 3

In [ ]:
# Test text generation using the registered lab-chat model
conn = psycopg.connect(**DB_CONFIG)
with conn.cursor() as cur:
    cur.execute(
        "SELECT azure_ai.generate(%s, %s) AS generated_text;",
        ("Generate random text.", CHAT_MODEL_ALIAS),
    )
    generated_text = cur.fetchone()[0]
    print(generated_text)
conn.close()

## Task 2 Cell 4

In [ ]:
# Test a single-row embedding call using the registered lab-embedding model
import json

conn = psycopg.connect(**DB_CONFIG)
with conn.cursor() as cur:
    cur.execute(
        "SELECT azure_openai.create_embeddings(%s, %s)::vector AS embedding;",
        (EMBED_MODEL_ALIAS, "This is a test sentence for embedding."),
    )
    embedding = cur.fetchone()[0]

    if isinstance(embedding, str):
        embedding = json.loads(embedding)

    print(f"Embedding length: {len(embedding)}")
    print(f"First 10 values: {embedding[:10]}")
conn.close()

## Task 3 Cell 1

In [ ]:
# Initialize dataset: create tables and load cases from CSV
csv_path = env_path.parent.parent / "data" / "cases.csv"
if not csv_path.exists():
    raise FileNotFoundError(
        f"Could not find the lab dataset at {csv_path}. "
        "Confirm that data/cases.csv exists in the repository."
    )
print(f"CSV path: {csv_path}")

conn = psycopg.connect(**DB_CONFIG)
conn.autocommit = True

with conn.cursor() as cur:
    # Drop existing tables
    cur.execute("DROP TABLE IF EXISTS cases;")
    cur.execute("DROP TABLE IF EXISTS temp_cases;")

    # Create the cases table
    cur.execute("""
        CREATE TABLE cases(
            id SERIAL PRIMARY KEY,
            name TEXT,
            decision_date DATE,
            court_level TEXT,
            opinion TEXT
        );
    """)

    # Create temp staging table and load CSV
    cur.execute("CREATE TABLE temp_cases(data jsonb);")

    with open(csv_path, "r", encoding="utf-8") as f:
        with cur.copy("COPY temp_cases (data) FROM STDIN WITH (FORMAT csv, HEADER true)") as copy:
            while chunk := f.read(8192):
                copy.write(chunk)

    print("CSV loaded into temp_cases.")

    # Insert parsed data into cases table
    cur.execute("""
        INSERT INTO cases
        SELECT
            (data#>>'{id}')::int AS id,
            (data#>>'{name_abbreviation}')::text AS name,
            (data#>>'{decision_date}')::date AS decision_date,
            (data#>>'{court,name}')::text AS court_level,
            array_to_string(
                ARRAY(
                    SELECT jsonb_path_query(
                        data, '$.casebody.opinions[*].text'
                    ) #>> '{}'
                ),
                E'\n\n'
            ) AS opinion
        FROM temp_cases;
    """)

    cur.execute("""
        SELECT setval(
            pg_get_serial_sequence('cases', 'id'),
            COALESCE(max(id), 1),
            max(id) IS NOT NULL
        )
        FROM cases;
    """)

    cur.execute("SELECT count(*) FROM cases;")
    count = cur.fetchone()[0]
    print(f"Done. {count} cases loaded into the cases table.")

conn.close()

## Task 3 Cell 2

In [ ]:
# Test query: select top 5 rows from cases
conn = psycopg.connect(**DB_CONFIG)
with conn.cursor() as cur:
    cur.execute("SELECT * FROM cases LIMIT 5;")
    rows = cur.fetchall()
    for row in rows:
        print(row)
conn.close()

## Task 4 Cell 1

In [ ]:
# Reset and create the AI Pipeline sink table.
conn = psycopg.connect(**DB_CONFIG)
conn.autocommit = True

with conn.cursor() as cur:
    try:
        cur.execute("SELECT ai.drop_pipeline(%s);", (PIPELINE_NAME,))
        print(f"Dropped existing pipeline '{PIPELINE_NAME}'.")
    except Exception as e:
        print(f"No existing pipeline to drop ({str(e).splitlines()[0]}).")

    cur.execute(f"DROP TABLE IF EXISTS {CHUNK_TABLE} CASCADE;")
    cur.execute(f"""
        CREATE TABLE {CHUNK_TABLE} (
            doc_id       INT,
            chunk_index  INT,
            chunk_text   TEXT,
            embedding    vector({EMBED_DIMS}),
            metadata     JSONB,
            court_level  TEXT,
            decision_date DATE
        );
    """)
    print(f"Created sink table {CHUNK_TABLE}.")

    cur.execute(f"""
        CREATE INDEX idx_case_opinion_chunks_doc_id
        ON {CHUNK_TABLE} (doc_id);
    """)
    cur.execute(f"""
        CREATE INDEX idx_case_opinion_chunks_court_date
        ON {CHUNK_TABLE} (court_level, decision_date);
    """)
    print("Created btree indexes for source linkage and structured filters.")

    cur.execute(
        """
        SELECT a.attname, format_type(a.atttypid, a.atttypmod) AS coltype
        FROM pg_attribute a
        WHERE a.attrelid = %s::regclass
          AND a.attnum > 0
          AND NOT a.attisdropped
        ORDER BY a.attnum;
        """,
        (CHUNK_TABLE,),
    )
    print("\nSink table columns:")
    for name, coltype in cur.fetchall():
        print(f"  {name:<14} {coltype}")

conn.close()
print(f"\nDone. {CHUNK_TABLE} is ready for the AI Pipeline.")

## Task 4 Cell 2

In [ ]:
POLL_SECONDS = 10
POLL_TIMEOUT_S = 1800
TERMINAL_STATES = {"completed", "succeeded", "failed", "cancelled", "error"}

def show_status(cur):
    """Print the preview status columns and return the latest status."""
    cur.execute("SELECT * FROM ai.status(%s);", (PIPELINE_NAME,))
    columns = [column.name for column in cur.description]
    rows = cur.fetchall()
    if not rows:
        print("  (ai.status returned no rows yet)")
        return {}
    latest = dict(zip(columns, rows[0]))
    print("  " + "  ".join(f"{key}={value}" for key, value in latest.items()))
    return latest

with psycopg.connect(**DB_CONFIG, autocommit=True) as conn, conn.cursor() as cur:
    cur.execute(f"SELECT count(*) FROM {CHUNK_TABLE};")
    existing_chunks = cur.fetchone()[0]
    if existing_chunks:
        status = show_status(cur)
        state = str(status.get("last_run_status", "")).lower()
        if state not in {"completed", "succeeded"}:
            raise RuntimeError(
                f"Existing sink has {existing_chunks} chunks but pipeline state is {state!r}. "
                "Wait for completion or inspect the failed run before retrying."
            )
        print(f"Reusing {existing_chunks} existing chunks; no embeddings will be regenerated.")
    else:
        cur.execute(
            """
            SELECT ai.create_pipeline(
                name => %(name)s,
                source => ai.table_source(table_name => 'cases', incremental_column => 'id', schema_name => 'public'),
                steps => ARRAY[
                    ai.chunk_step(
                        input => 'opinion', 
                        chunk_size => 512, 
                        method => 'token',
                        overlap => 64),
                    ai.embed_step(
                        input => 'chunk_text', 
                        model => %(embed_model)s, 
                        dimensions => %(dims)s)
                ],
                trigger => 'manual',
                sink => ai.table_sink(
                            table_name => 'case_opinion_chunks', 
                            schema_name => 'public')
            );
            """,
            {"name": PIPELINE_NAME, "embed_model": EMBED_MODEL_ALIAS, "dims": EMBED_DIMS},
        )
        print(f"Created pipeline '{PIPELINE_NAME}'.")

        try:
            cur.execute("SELECT ai.explain(%s);", (PIPELINE_NAME,))
            print("\nCompiled pipeline definition:")
            for row in cur.fetchall():
                print(f"  {row[0]}")
        except Exception as error:
            print(f"\n(ai.explain unavailable: {str(error).splitlines()[0]})")

        cur.execute("SELECT ai.run(%s);", (PIPELINE_NAME,))
        print(f"\nStarted initial run of '{PIPELINE_NAME}'.")

        start = time.monotonic()
        while True:
            elapsed = int(time.monotonic() - start)
            print(f"\n[{elapsed}s] ai.status('{PIPELINE_NAME}'):")
            status = show_status(cur)
            cur.execute(f"SELECT count(*), count(embedding) FROM {CHUNK_TABLE};")
            chunk_rows, embedded_rows = cur.fetchone()
            print(f"  sink rows={chunk_rows}  rows with embedding={embedded_rows}")

            state = str(status.get("last_run_status", "")).lower()
            if state in TERMINAL_STATES:
                break
            if elapsed >= POLL_TIMEOUT_S:
                raise TimeoutError(
                    "Stopped polling; the pipeline may still be running. "
                    "Check ai.status() and rerun this cell after completion to enrich its output."
                )
            time.sleep(POLL_SECONDS)

        if state not in {"completed", "succeeded"}:
            raise RuntimeError(
                f"Pipeline ended with state {state!r}; output was not enriched. "
                "Inspect ai.pipeline_runs.error before rebuilding with Parts 2.8 through 2.10."
            )

## Task 4 Cell 3

In [ ]:
def enrich_pipeline_output(cur):
    """Map whitespace-normalized chunks uniquely before updating source IDs."""
    mapping_cte = f"""
        WITH probes AS MATERIALIZED (
            SELECT DISTINCT ON (doc_id)
                   doc_id AS pipeline_doc_id,
                   btrim(regexp_replace(chunk_text, '[[:space:]]+', ' ', 'g')) AS normalized
            FROM {CHUNK_TABLE}
            ORDER BY doc_id, chunk_index
        ),
        sources AS MATERIALIZED (
            SELECT id, court_level, decision_date,
                   btrim(regexp_replace(opinion, '[[:space:]]+', ' ', 'g')) AS normalized
            FROM {SOURCE_TABLE}
        ),
        mapping AS MATERIALIZED (
            SELECT probes.pipeline_doc_id, sources.id AS case_id,
                   sources.court_level, sources.decision_date, sources.normalized
            FROM probes
            JOIN sources ON probes.normalized <> ''
              AND position(probes.normalized IN sources.normalized) > 0
        )
    """
    with cur.connection.transaction():
        cur.execute(f"LOCK TABLE {SOURCE_TABLE} IN SHARE MODE;")
        cur.execute(f"LOCK TABLE {CHUNK_TABLE} IN SHARE ROW EXCLUSIVE MODE;")
        cur.execute(mapping_cte + f"""
            SELECT count(DISTINCT pipeline_doc_id), count(*), count(DISTINCT case_id),
                   (SELECT count(DISTINCT doc_id) FROM {CHUNK_TABLE}),
                   (SELECT count(*) FROM {SOURCE_TABLE})
            FROM mapping;
        """)
        mapped_docs, mapping_rows, mapped_cases, pipeline_docs, source_cases = cur.fetchone()
        if not (0 < mapped_docs == mapping_rows == mapped_cases == pipeline_docs == source_cases):
            raise RuntimeError(
                "Could not uniquely map all pipeline documents to all source cases: "
                f"mapped={mapped_docs}, candidates={mapping_rows}, cases={mapped_cases}, "
                f"expected={pipeline_docs}, source_cases={source_cases}. No rows changed."
            )

        cur.execute(mapping_cte + f"""
            SELECT count(*)
            FROM {CHUNK_TABLE} AS chunks
            LEFT JOIN mapping ON chunks.doc_id = mapping.pipeline_doc_id
            WHERE mapping.case_id IS NULL
               OR chunks.chunk_index IS NULL
               OR chunks.chunk_text IS NULL
               OR btrim(regexp_replace(chunks.chunk_text, '[[:space:]]+', ' ', 'g')) = ''
               OR position(
                   btrim(regexp_replace(chunks.chunk_text, '[[:space:]]+', ' ', 'g'))
                   IN mapping.normalized
               ) = 0;
        """)
        invalid_chunks = cur.fetchone()[0]
        if invalid_chunks:
            raise RuntimeError(
                f"{invalid_chunks} chunks do not match their mapped source opinion. No rows changed."
            )

        cur.execute(mapping_cte + f"""
            UPDATE {CHUNK_TABLE} AS chunks
            SET doc_id = mapping.case_id,
                metadata = jsonb_build_object('source_case_id', mapping.case_id),
                court_level = mapping.court_level,
                decision_date = mapping.decision_date
            FROM mapping
            WHERE chunks.doc_id = mapping.pipeline_doc_id;
        """)
        enriched_rows = cur.rowcount
    print(f"Enriched {enriched_rows} chunks with verified source case IDs and filter metadata.")

with psycopg.connect(**DB_CONFIG, autocommit=True) as conn, conn.cursor() as cur:
    enrich_pipeline_output(cur)
    cur.execute(f"SELECT doc_id, chunk_index, LEFT(chunk_text, 100), embedding, metadata, court_level, decision_date FROM {CHUNK_TABLE} LIMIT 5;")
    rows = cur.fetchall()

    print("\n Preview of enriched chunks:")
    for row in rows:
        print(row)

print("\nDone. Pipeline completed and all chunks enriched. Verify it in Part 2.10.")


## Task 4 Cell 4

In [ ]:
# Verify the AI Pipeline output. Preview pipelines can report success while
# silently dropping a column, so check the invariants explicitly.
checks = []


def record(ok, label, detail=""):
    checks.append(bool(ok))
    print(f"{'✅' if ok else '❌'} {label}" + (f"\n     -> {detail}" if detail else ""))


conn = psycopg.connect(**DB_CONFIG)
conn.autocommit = True

with conn.cursor() as cur:
    cur.execute(f"SELECT count(*) FROM {SOURCE_TABLE};")
    source_cases = cur.fetchone()[0]
    record(source_cases > 0, f"Source rows in {SOURCE_TABLE}: {source_cases}")

    cur.execute(f"SELECT count(*), count(DISTINCT doc_id) FROM {CHUNK_TABLE};")
    chunk_rows, cases_with_chunks = cur.fetchone()
    record(
        chunk_rows > 0,
        f"Chunk rows in {CHUNK_TABLE}: {chunk_rows} across {cases_with_chunks} cases",
        "" if chunk_rows else "the pipeline produced no output - re-check Part 2.9",
    )

    if chunk_rows:
        cur.execute(f"""
            SELECT min(cnt), round(avg(cnt), 2), max(cnt)
            FROM (SELECT count(*) AS cnt FROM {CHUNK_TABLE} GROUP BY doc_id) t;
        """)
        lo, avg, hi = cur.fetchone()
        print(f"     chunks per case: min={lo}  avg={avg}  max={hi}")

    cur.execute(f"SELECT count(*) FROM {CHUNK_TABLE} WHERE embedding IS NULL;")
    null_embeddings = cur.fetchone()[0]
    record(
        null_embeddings == 0,
        f"Chunks missing an embedding: {null_embeddings}",
        "" if null_embeddings == 0 else "check the sink embedding column and ai.embed step",
    )

    cur.execute(f"""
        SELECT DISTINCT vector_dims(embedding)
        FROM {CHUNK_TABLE}
        WHERE embedding IS NOT NULL;
    """)
    dims = sorted(r[0] for r in cur.fetchall())
    record(
        dims == [EMBED_DIMS],
        f"Embedding dimensionality: {dims or 'none'}",
        "" if dims == [EMBED_DIMS] else f"expected exactly [{EMBED_DIMS}]",
    )

    cur.execute(f"""
        SELECT count(*) FROM {CHUNK_TABLE}
        WHERE chunk_text IS NULL OR btrim(chunk_text) = '';
    """)
    empty_text = cur.fetchone()[0]
    record(empty_text == 0, f"Chunks with empty chunk_text: {empty_text}")

    cur.execute(f"""
        SELECT count(*) FROM {CHUNK_TABLE}
        WHERE court_level IS NULL OR decision_date IS NULL;
    """)
    missing_filters = cur.fetchone()[0]
    record(
        missing_filters == 0,
        f"Chunks missing court/date filter metadata: {missing_filters}",
        "" if missing_filters == 0 else "source columns did not land in the sink; inspect the pipeline definition",
    )

    cur.execute(f"""
        SELECT count(*) FROM {SOURCE_TABLE} c
        WHERE NOT EXISTS (SELECT 1 FROM {CHUNK_TABLE} k WHERE k.doc_id = c.id);
    """)
    unchunked = cur.fetchone()[0]
    record(
        unchunked == 0,
        f"Source cases with no pipeline output: {unchunked}",
        "" if unchunked == 0 else "rerun Parts 2.8 through 2.10 to rebuild and enrich the sink",
    )

    cur.execute(f"""
        SELECT count(*) FROM {CHUNK_TABLE} k
        LEFT JOIN {SOURCE_TABLE} c ON c.id = k.doc_id
        WHERE c.id IS NULL;
    """)
    orphans = cur.fetchone()[0]
    record(orphans == 0, f"Orphaned chunks: {orphans}")

    cur.execute(f"""
        SELECT count(*)
        FROM {CHUNK_TABLE} k
        JOIN {SOURCE_TABLE} c ON c.id = k.doc_id
        WHERE k.court_level IS DISTINCT FROM c.court_level
           OR k.decision_date IS DISTINCT FROM c.decision_date;
    """)
    metadata_mismatches = cur.fetchone()[0]
    record(
        metadata_mismatches == 0,
        f"Chunks whose filter metadata differs from public.cases: {metadata_mismatches}",
    )

    try:
        cur.execute("SELECT * FROM ai.status(%s);", (PIPELINE_NAME,))
        columns = [d.name for d in cur.description]
        row = cur.fetchone()
        status = dict(zip(columns, row)) if row else {}
        state = str(status.get("last_run_status", "unknown")).lower()
        record(
            state in {"completed", "succeeded"},
            f"Pipeline last_run_status: {state}",
            "" if state in {"completed", "succeeded"} else str(status),
        )
    except Exception as e:
        record(False, "Could not read ai.status()", str(e).splitlines()[0])

    print("\n// Sample chunk-level semantic search (registered lab-embedding model) //")
    try:
        cur.execute(
            f"""
            WITH q AS MATERIALIZED (
                SELECT azure_openai.create_embeddings(%s, %s)::vector AS qv
            )
            SELECT k.doc_id, c.name, k.chunk_index,
                   k.embedding <=> (SELECT qv FROM q) AS distance,
                   left(replace(k.chunk_text, E'\n', ' '), 120) AS snippet
            FROM {CHUNK_TABLE} k
            JOIN {SOURCE_TABLE} c ON c.id = k.doc_id
            WHERE k.decision_date >= DATE '1900-01-01'
            ORDER BY k.embedding <=> (SELECT qv FROM q)
            LIMIT 5;
            """,
            (EMBED_MODEL_ALIAS, "the city regraded the road and now the property floods"),
        )
        for doc_id, name, idx, distance, snippet in cur.fetchall():
            print(f"  case {doc_id} | chunk {idx} | distance={distance:.4f} | {name}: {snippet}...")
    except Exception as e:
        print(f"  ⚠️  Sample search failed: {str(e).splitlines()[0]}")

conn.close()

print("")
if all(checks):
    print("Done. All pipeline checks passed - the sink is ready for indexing in Part 2.13.")
else:
    print("Done. One or more checks FAILED - fix them before moving on to Notebook 2.")

## Task 5 Cell 1

In [ ]:
# Build the case citation graph using Apache AGE
import time

import psycopg
from psycopg.types.json import Jsonb

CASE_GRAPH_NAME = "case_graph"
CASE_GRAPH_PIPELINE_NAME = "case_graph_load"
CASE_GRAPH_ONTOLOGY = {
    "nodes": [
        {
            "label": "case",
            "identity": "case_id",
            "properties": ["case_id", "name", "decision_date", "court_level", "opinion"],
        }
    ],
    "edges": [{"label": "REF", "from": "case", "to": "case"}],
}
CASE_GRAPH_MAPPING = {
    "nodes": [
        {"label": "case", "property": "case_id", "from": "case_id"},
        {"label": "case", "property": "name", "from": "name"},
        {"label": "case", "property": "decision_date", "from": "decision_date"},
        {"label": "case", "property": "court_level", "from": "court_level"},
        {"label": "case", "property": "opinion", "from": "opinion"},
    ],
    "each": [
        {
            "path": "citations",
            "edges": [
                {
                    "edge_label": "REF",
                    "from_key": "$.case_id",
                    "to_key": "@.case_id",
                }
            ],
        }
    ],
}

# Shape the existing structured data; do not ask an LLM to rediscover citations.
CASE_GRAPH_DOCUMENTS_SQL = """
    WITH citation_pairs AS (
        SELECT DISTINCT
            raw.data ->> 'id' AS id_from,
            target.data ->> 'id' AS id_to
        FROM public.temp_cases raw
        CROSS JOIN LATERAL
            jsonb_array_elements(raw.data -> 'cites_to') AS citation
        CROSS JOIN LATERAL
            jsonb_array_elements_text(citation -> 'case_ids') AS cited(case_id)
        JOIN public.temp_cases target ON target.data ->> 'id' = cited.case_id
    ),
    citation_lists AS (
        SELECT id_from,
               jsonb_agg(
                   jsonb_build_object('case_id', id_to) ORDER BY id_to
               ) AS citations
        FROM citation_pairs
        GROUP BY id_from
    )
    SELECT c.id::text AS case_id,
           jsonb_build_object(
               'case_id', c.id::text,
               'name', c.name,
               'decision_date', c.decision_date::text,
               'court_level', c.court_level,
               'opinion', c.opinion,
               'citations', COALESCE(refs.citations, '[]'::jsonb)
           ) AS document
    FROM public.cases c
    LEFT JOIN citation_lists refs ON refs.id_from = c.id::text
"""
CASE_GRAPH_SOURCE_SQL = """
    CREATE TABLE IF NOT EXISTS public.case_graph_source (
        graph_row_id bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        case_id text NOT NULL UNIQUE,
        document jsonb NOT NULL CHECK (jsonb_typeof(document) = 'object'),
        CHECK (document ->> 'case_id' = case_id)
    );
"""
CASE_GRAPH_CREATE_SQL = """
    SELECT ai.create_graph(
        name => %s,
        ontology => %s
    );
"""
CASE_GRAPH_DESCRIPTORS_SQL = """
    SELECT ai.table_source(
               table_name => 'case_graph_source',
               incremental_column => 'graph_row_id',
               schema_name => 'public'
           ),
           ai.graph_sink(
               graph_name => %s,
               mappings => %s,
               source_column => 'document'
           );
"""
CASE_GRAPH_REGISTER_SQL = """
    SELECT ai.create_pipeline(
        name => %s,
        source => ai.table_source(
            table_name => 'case_graph_source',
            incremental_column => 'graph_row_id',
            schema_name => 'public'
        ),
        steps => ARRAY[]::jsonb[],
        sink => ai.graph_sink(
            graph_name => %s,
            mappings => %s,
            source_column => 'document'
        ),
        trigger => 'manual'
    );
"""
CASE_GRAPH_RUNTIME_SQL = """
    SELECT role.rolsuper,
           to_regprocedure('ai._graph_sink_load(bigint,bigint)') IS NOT NULL,
           EXISTS (SELECT 1 FROM pg_extension WHERE extname = 'pg_durable'),
           current_database() = current_setting('pg_durable.database', true)
    FROM pg_roles role
    WHERE role.rolname = current_user;
"""
CASE_GRAPH_STALE_SQL = f"""
    WITH expected AS ({CASE_GRAPH_DOCUMENTS_SQL})
    SELECT existing.case_id
    FROM public.case_graph_source existing
    LEFT JOIN expected USING (case_id)
    WHERE expected.case_id IS NULL
       OR existing.document IS DISTINCT FROM expected.document
    LIMIT 1;
"""
CASE_GRAPH_APPEND_SQL = f"""
    INSERT INTO public.case_graph_source (case_id, document)
    SELECT expected.case_id, expected.document
    FROM ({CASE_GRAPH_DOCUMENTS_SQL}) expected
    WHERE NOT EXISTS (
        SELECT 1 FROM public.case_graph_source existing
        WHERE existing.case_id = expected.case_id
    )
    ORDER BY expected.case_id::integer;
"""

with psycopg.connect(**DB_CONFIG, autocommit=True) as conn, conn.cursor() as cur:
    cur.execute("SET search_path = public, ag_catalog, \"$user\";")
    cur.execute("SELECT pg_try_advisory_lock(hashtext(%s));", (CASE_GRAPH_PIPELINE_NAME,))
    if not cur.fetchone()[0]:
        raise RuntimeError("Another case graph cell is running. Wait for it to finish.")

    cur.execute("""
        SELECT required.signature
        FROM unnest(ARRAY[
            'ai.create_graph(text,jsonb)',
            'ai.graph_sink(text,jsonb,text)',
            'ai.table_source(text,text,text)',
            'ai.create_pipeline(text,jsonb,jsonb[],jsonb,text,jsonb)',
            'ai.run(text)', 'ai.result(text)'
        ]) AS required(signature)
        WHERE to_regprocedure(required.signature) IS NULL;
    """)
    missing = [row[0] for row in cur.fetchall()]
    if missing:
        raise RuntimeError("This HorizonDB build is missing: " + ", ".join(missing))
    cur.execute("SELECT 1 FROM pg_extension WHERE extname = 'age';")
    if cur.fetchone() is None:
        raise RuntimeError("AGE is not installed. Run Part 2.4.")
    cur.execute(CASE_GRAPH_RUNTIME_SQL)
    is_superuser, durable_loader, durable_installed, worker_database_matches = cur.fetchone()
    if is_superuser:
        raise RuntimeError("ai.run rejects superusers. Connect as the nonsuperuser lab role.")
    if durable_loader and (not durable_installed or not worker_database_matches):
        raise RuntimeError(
            "This build uses durable graph loading. Have the administrator enable "
            "pg_durable and configure its workers for this lab database before continuing."
        )

    cur.execute("""
        SELECT (SELECT count(*) FROM public.cases),
               (SELECT count(*) FROM public.temp_cases),
               (SELECT count(DISTINCT data ->> 'id') FROM public.temp_cases),
               EXISTS (
                   SELECT 1
                   FROM public.cases c
                   FULL JOIN public.temp_cases raw ON raw.data ->> 'id' = c.id::text
                   WHERE c.id IS NULL OR raw.data IS NULL
               );
    """)
    case_count, raw_count, distinct_ids, mismatched_ids = cur.fetchone()
    if not case_count or case_count != raw_count or raw_count != distinct_ids or mismatched_ids:
        raise RuntimeError("cases and temp_cases must contain the same unique case IDs. Run Part 2.6.")

    cur.execute("SELECT ontology FROM ai.graphs WHERE name = %s;", (CASE_GRAPH_NAME,))
    registered_graph = cur.fetchone()
    if registered_graph is None:
        cur.execute("SELECT 1 FROM ag_catalog.ag_graph WHERE name = %s;", (CASE_GRAPH_NAME,))
        if cur.fetchone() is not None:
            raise RuntimeError(
                "case_graph already exists outside this pipeline. Use a fresh lab database "
                "or an explicitly reviewed reset; this cell will not delete it."
            )
        cur.execute(CASE_GRAPH_CREATE_SQL, (CASE_GRAPH_NAME, Jsonb(CASE_GRAPH_ONTOLOGY)))
    elif registered_graph[0] != CASE_GRAPH_ONTOLOGY:
        raise RuntimeError("Existing case_graph has a different ontology; it was not changed.")

    cur.execute(CASE_GRAPH_SOURCE_SQL)
    cur.execute("""
        SELECT attname, format_type(atttypid, atttypmod), attidentity::text
        FROM pg_attribute
        WHERE attrelid = 'public.case_graph_source'::regclass
          AND attnum > 0 AND NOT attisdropped
        ORDER BY attnum;
    """)
    if cur.fetchall() != [
        ("graph_row_id", "bigint", "a"), ("case_id", "text", ""), ("document", "jsonb", "")
    ]:
        raise RuntimeError("Existing case_graph_source has an incompatible schema.")

    cur.execute(
        CASE_GRAPH_DESCRIPTORS_SQL, (CASE_GRAPH_NAME, Jsonb(CASE_GRAPH_MAPPING))
    )
    expected_source, expected_sink = cur.fetchone()
    cur.execute("""
        SELECT source_config, steps, sink_config, trigger_type, paused
        FROM ai.pipelines WHERE name = %s;
    """, (CASE_GRAPH_PIPELINE_NAME,))
    registered_pipeline = cur.fetchone()
    if registered_pipeline is None:
        cur.execute(
            CASE_GRAPH_REGISTER_SQL,
            (CASE_GRAPH_PIPELINE_NAME, CASE_GRAPH_NAME, Jsonb(CASE_GRAPH_MAPPING)),
        )
    elif registered_pipeline != (expected_source, [], expected_sink, "manual", False):
        raise RuntimeError("Existing case_graph_load has a different or paused definition.")

    cur.execute("""
        SELECT run_id, status
        FROM ai.result(%s) ORDER BY run_id DESC LIMIT 1;
    """, (CASE_GRAPH_PIPELINE_NAME,))
    previous_run = cur.fetchone()
    previous_run_id = previous_run[0] if previous_run else 0
    if previous_run and previous_run[1] not in {"completed", "failed", "cancelled"}:
        raise RuntimeError("A previous graph run is not terminal. Wait or inspect ai.result().")

    with conn.transaction():
        cur.execute(CASE_GRAPH_STALE_SQL)
        changed_case = cur.fetchone()
        if changed_case:
            raise RuntimeError(
                f"Case {changed_case[0]} or its citations changed. This append-only example "
                "does not retract old graph facts; review reconciliation before continuing."
            )
        cur.execute(CASE_GRAPH_APPEND_SQL)
        print(f"Prepared {cur.rowcount} new cases for incremental graph loading.")

    cur.execute("SELECT ai.run(%s);", (CASE_GRAPH_PIPELINE_NAME,))
    print(f"Dispatched {CASE_GRAPH_PIPELINE_NAME}: {cur.fetchone()[0]}")
    deadline = time.monotonic() + 1800
    while True:
        cur.execute("""
            SELECT run_id, status, rows_processed, error
            FROM ai.result(%s)
            WHERE run_id > %s
            ORDER BY run_id DESC LIMIT 1;
        """, (CASE_GRAPH_PIPELINE_NAME, previous_run_id))
        run = cur.fetchone()
        if run:
            print(f"Run {run[0]}: {run[1]}; source rows processed: {run[2]}")
            if run[1] == "completed":
                break
            if run[1] in {"failed", "cancelled"}:
                raise RuntimeError(f"Graph run {run[0]} ended {run[1]}: {run[3]}")
        if time.monotonic() >= deadline:
            raise TimeoutError(
                "Stopped polling; the graph run may still be active. Inspect ai.result() "
                "before rerunning or resetting anything."
            )
        time.sleep(5)

    cur.execute("""
        SELECT count(*), COALESCE(sum(jsonb_array_length(document -> 'citations')), 0)
        FROM public.case_graph_source;
    """)
    expected_nodes, expected_edges = cur.fetchone()
    cur.execute("""
        SELECT node_count
        FROM ag_catalog.cypher('case_graph', $$
            MATCH (n:case) RETURN count(n)
        $$) AS (node_count ag_catalog.agtype);
    """)
    actual_nodes = int(str(cur.fetchone()[0]))
    cur.execute("""
        SELECT edge_count
        FROM ag_catalog.cypher('case_graph', $$
            MATCH (:case)-[e:REF]->(:case) RETURN count(e)
        $$) AS (edge_count ag_catalog.agtype);
    """)
    actual_edges = int(str(cur.fetchone()[0]))
    if (actual_nodes, actual_edges) != (expected_nodes, expected_edges):
        raise RuntimeError(
            f"Graph counts differ: expected {expected_nodes} nodes / {expected_edges} edges, "
            f"got {actual_nodes} / {actual_edges}. Inspect the graph; do not reset automatically."
        )
    print(f"Done. case_graph is ready: {actual_nodes} case nodes, {actual_edges} unique REF edges.")


## Task 6 Cell 1

In [ ]:
conn = psycopg.connect(**DB_CONFIG)
conn.autocommit = True

with conn.cursor() as cur:
    # Part 2.4 verifies installation and the required preload setting.
    # Check the catalog here without issuing redundant CREATE EXTENSION DDL.
    cur.execute("SELECT extversion FROM pg_extension WHERE extname = 'pg_textsearch';")
    row = cur.fetchone()
    print(f"pg_textsearch extension is ready. Version: {row[0]}" if row
          else "pg_textsearch NOT installed")

    # BM25 indexes are single-column. Index the opinion text (the main search target).
    cur.execute("DROP INDEX IF EXISTS idx_cases_bm25;")
    cur.execute("""
        CREATE INDEX idx_cases_bm25
        ON public.cases
        USING bm25 (opinion) WITH (text_config = 'english');
    """)
    print("BM25 index created on cases.opinion")

    # Query with the <@> operator. It returns a BM25 score where LOWER = more
    # relevant, so ORDER BY ascending and take the top-k.
    cur.execute("""
        SELECT id, name, opinion <@> 'water leaking' AS bm25_score
        FROM cases
        ORDER BY opinion <@> 'water leaking'
        LIMIT 5;
    """)
    matches = cur.fetchall()
    print(f"Top {len(matches)} cases for 'water leaking':")
    for m in matches:
        print(f"  [{m[0]}] {m[1]} — score {m[2]:.4f}")

conn.close()

## Task 6 Cell 2

In [ ]:
DISKANN_INDEX_NAME = "idx_case_opinion_chunks_diskann"

with psycopg.connect(**DB_CONFIG, autocommit=True) as conn, conn.cursor() as cur:
    cur.execute("""
        SELECT extname
        FROM pg_extension
        WHERE extname IN ('vector', 'pg_diskann');
    """)
    installed_vector_extensions = {row[0] for row in cur.fetchall()}
    missing_vector_extensions = {
        'vector', 'pg_diskann'
    } - installed_vector_extensions
    if missing_vector_extensions:
        raise RuntimeError(
            'Missing required extensions: '
            + ', '.join(sorted(missing_vector_extensions))
            + '. Run Part 2.4 before building the DiskANN index.'
        )
    cur.execute(f"DROP INDEX IF EXISTS {DISKANN_INDEX_NAME};")

    build_variants = [
        (
            "spherical quantization",
            f"""
            CREATE INDEX {DISKANN_INDEX_NAME}
            ON {CHUNK_TABLE}
            USING diskann (embedding vector_cosine_ops)
            WITH (spherical_quantized = true);
            """,
        ),
        (
            "full precision",
            f"""
            CREATE INDEX {DISKANN_INDEX_NAME}
            ON {CHUNK_TABLE}
            USING diskann (embedding vector_cosine_ops);
            """,
        ),
    ]

    built = None
    for label, ddl in build_variants:
        try:
            cur.execute(ddl)
            built = label
            break
        except Exception as e:
            print(f"  {label} build unavailable: {str(e).splitlines()[0]}")

    if built is None:
        raise RuntimeError(
            f"Could not build a DiskANN index on {CHUNK_TABLE}(embedding). "
            "Confirm pg_diskann is installed and Part 2.9 populated the sink."
        )

    cur.execute(
        """
        SELECT c.relname, i.indisvalid, i.indisready, pg_get_indexdef(i.indexrelid)
        FROM pg_index i
        JOIN pg_class c ON c.oid = i.indexrelid
        WHERE c.relname = %s;
        """,
        (DISKANN_INDEX_NAME,),
    )
    index_name, is_valid, is_ready, definition = cur.fetchone()
    print(f"Index build mode: {built}")
    print(f"Index: {index_name}  valid={is_valid}  ready={is_ready}")
    print(definition)

print(f"\npg_diskann ready: {DISKANN_INDEX_NAME} on {CHUNK_TABLE}(embedding)")